# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aun-Mehdi117/-Flyrank-ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Classification, whose output doubles as a ranking score.**

Lane 2 (Refresh / Content Opportunity Scoring) sounds like a ranking problem on the surface —
"which pages should an editor look at first?" — but the thing I'd actually train a model to
predict is a yes/no outcome: **is this page declining?** (`is_declining_label`, derived from
`trend_direction`). That makes the training task binary classification.

The ranking comes *after* classification, for free: instead of thresholding the model's output
at 0.5, I sort every page by its predicted probability of decline and hand editors the top of
that sorted list as their weekly queue. So the task type is classification; the **product** is a
ranked queue. This matches the mapping in the `framing-ml-problems` skill table almost exactly
("Will this one decline?" → classification, label from an observed outcome), and it's also why
`scripts/03_train_model.py` in this repo trains `LogisticRegression` / `DecisionTreeClassifier` /
`RandomForestClassifier` rather than a learning-to-rank model — the starter pipeline already made
this same call.


In [1]:
# Setup: load the starter slice once, reuse it in every section below.
import pandas as pd

pd.set_option("display.max_columns", 12)
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print(f"rows: {len(df):,}  columns: {df.shape[1]}  unique clients: {df.client_id.nunique()}")


rows: 30,000  columns: 44  unique clients: 32


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target: `is_declining_label`** — 1 when `trend_direction == "down"`, else 0. This is what
`scripts/01_prepare_features.py` adds to the raw CSV, and it's the label I'd train against.

Where it comes from, honestly: `trend_direction` is itself computed from `trend_pct`
(`(impressions_last_30d − impressions_prev_30d) / impressions_prev_30d`), bucketed with a fixed
**±20% threshold** into `up` / `down` / `stable` / `flat` / `new`. So this is a **hybrid** —
the magnitude (`trend_pct`) is an observed measurement of real traffic, but the *label* is a
rule I (well, the pipeline) chose applied on top of it. That distinction matters for two reasons:

1. **Leakage** — `trend_direction` and `trend_pct` can never be model *features*, only the
   target. They're computed from the same 30-day windows I'd be predicting into.
2. **Honesty about the threshold** — "declining" here means "dropped >20% in impressions,"
   not "editorially declining." A page that dropped 19% or is flat-but-low-demand is invisible
   to this exact label, which is a real limitation I'll need to name later (see the lane guide's
   note on decline vs. consolidation, seasonality, and noise).

So it's a **defined-rule proxy for a real, observed outcome** — not something I invented from
nothing, but not a clean ground truth either.


In [2]:
# The label, and the leakage check the flyrank-data skill flags explicitly.
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("trend_direction value counts:")
print(df["trend_direction"].value_counts())
print()
print("is_declining_label rate (the target's base rate):")
print(df["is_declining_label"].value_counts(normalize=True).round(3))
print()
print("Rows with trend_pct but no trend_direction (should be 0 - sanity check):",
      int((df["trend_pct"].notna() & df["trend_direction"].isna()).sum()))


trend_direction value counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

is_declining_label rate (the target's base rate):
is_declining_label
1    0.542
0    0.458
Name: proportion, dtype: float64

Rows with trend_pct but no trend_direction (should be 0 - sanity check): 0


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary: precision@K** (K = an editor's weekly review capacity, e.g. top 25–50 pages per
client). This is the metric that matches the actual decision: an editor doesn't act on every
row, they act on a fixed-size queue, so what matters is "of the pages I put at the top, how
many were really declining?" — not overall accuracy across all 30,000 rows.

**Secondary: ROC-AUC**, to track ranking quality independent of any one K, and to compare model
versions during development.

**Why not plain accuracy:** the base rate for `is_declining_label` is **54.2%** (computed
below) — close enough to a coin flip that accuracy is a weak, easy-to-game number. A model that
just predicts the majority class already scores ~54%. Accuracy also doesn't reflect the cost
asymmetry from my Week 1 framing: missing a real decliner (false negative) is more expensive
than flagging a stable page (false positive), because a missed decliner sits unnoticed for
another week. Precision@K, evaluated at the *top* of the queue where editors actually spend
their time, is the number I can defend to a content strategist without translation.


In [3]:
# Why accuracy alone is a weak bar here: the base rate is already near 50/50.
base_rate = df["is_declining_label"].mean()
print(f"Base rate for is_declining_label: {base_rate:.1%}")
print(f"A model that always predicts 'declining' already scores ~{max(base_rate, 1-base_rate):.1%} accuracy.")
print("-> accuracy near this number tells me almost nothing; precision@K at a realistic queue")
print("   size (e.g. K=50 per client) is the number that actually reflects editor capacity.")


Base rate for is_declining_label: 54.2%
A model that always predicts 'declining' already scores ~54.2% accuracy.
-> accuracy near this number tells me almost nothing; precision@K at a realistic queue
   size (e.g. K=50 per client) is the number that actually reflects editor capacity.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content item (page), for one client, summarized over a trailing 90-day window.**
Not one client, not one day of traffic — one page's 90-day snapshot. `content_id` is unique per
row; `client_id` groups pages under the 32 pseudonymized clients they belong to (and is the
column I'd use for a client-grouped train/test split later, never as a feature).


In [4]:
# One row = one page. Show it for real, with the columns this lane actually cares about.
lane_view = df[[
    "content_id", "client_id", "content_type", "avg_position", "impression_tier",
    "trend_pct", "trend_direction", "is_declining_label",
]]
print(f"lane_view: {lane_view.shape[0]:,} rows x {lane_view.shape[1]} columns, "
      f"{lane_view.content_id.nunique():,} unique content_id (== row count, confirms the grain)")
lane_view.head()


lane_view: 30,000 rows x 8 columns, 30,000 unique content_id (== row count, confirms the grain)


,content_id,client_id,content_type,avg_position,impression_tier,trend_pct,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,10.6,good,-41.4,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,20.3,good,-57.7,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,36.5,good,-60.9,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,6.2,good,-13.8,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,44.0,good,-34.7,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

I could write a single rule today: `if trend_direction == "down": flag it`. That's literally
what the label already is — so on its own it's circular, not a model. The real question is
whether a *simple, hand-tuned* rule using the other signals (position, demand, freshness, CTR)
would rank pages about as well as a model. My Week 1 numbers say no: three reasonable
rule-based candidate pools — "declining with demand" (43.8%), "page-one decay risk" (23.6%),
and "low-CTR-despite-visibility" (32.5%) — **overlap only partially**. A page can be in one,
two, or all three groups at once, and a single if/else chain forces me to pick one ordering of
those conditions and hard-code the cutoffs (why 100 impressions and not 80? why 180 days and not
150?). Those cutoffs would need constant re-tuning per client, since `position_tier`'s own note
in the data dictionary warns that the same tier means different things at different traffic
volumes.

That's the actual case for ML here: not that a rule is *impossible*, but that the signal lives
in the **combination and relative weighting** of several correlated-but-not-identical features
(demand, position, freshness, CTR, content type), and a model can learn client-appropriate
weights and interactions from data instead of me guessing thresholds by hand — while still
producing reason codes an editor can read, which is why the starter pipeline pairs the model
with an explainable baseline rather than replacing rules entirely.


In [5]:
# Quick evidence for "the signal is spread across correlated-but-not-identical features,
# not captured by one threshold": correlation of a few lane-relevant numeric signals with the
# label. None of them is a slam dunk on its own.
import numpy as np

df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
signals = ["log_impressions_90d", "avg_position", "ctr", "engagement_rate",
           "content_age_days", "days_since_last_update"]

corrs = df[signals + ["is_declining_label"]].corr(numeric_only=True)["is_declining_label"].drop("is_declining_label")
print("Correlation of individual signals with is_declining_label:")
print(corrs.sort_values(key=abs, ascending=False).round(3))
print()
print("No single signal is strongly correlated on its own (all well under ~0.2 in magnitude) —")
print("that's the case for combining several weak, partially-overlapping signals with a model")
print("rather than picking one and thresholding it.")


Correlation of individual signals with is_declining_label:
log_impressions_90d       0.177
content_age_days         -0.164
days_since_last_update    0.081
ctr                      -0.062
avg_position             -0.029
engagement_rate          -0.013
Name: is_declining_label, dtype: float64

No single signal is strongly correlated on its own (all well under ~0.2 in magnitude) —
that's the case for combining several weak, partially-overlapping signals with a model
rather than picking one and thresholding it.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.